In [ ]:
import sys, os
sys.path.insert(0, r"C:\Users\Utente\Desktop\progetto vscode\src")

In [1]:
import numpy as np
import networkx as nx
import pandas as pd
import ot
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
from networkx.algorithms import bipartite
import glob 
from functions import *
from skbio.stats.distance import DistanceMatrix, permanova, mantel, anosim
from scipy.spatial.distance import pdist, squareform


Funcion that reads a CSV file and returns a bipartite graph. 

In [2]:
def create_bipartite_graph(file_path):
    
    df = pd.read_csv(file_path, index_col=0, sep=None, engine='python')
   
    # Everything converted to numbers or into NaN 
    df = df.apply(pd.to_numeric, errors='coerce')
    # NaN transformed in 0
    df = df.fillna(0)

    # Creation of an empty graph
    G = nx.Graph()
    
    # Creations of the lis of the species of plants and pollinators
    plants = df.index.tolist()
    impollinators = df.columns.tolist()
    
    # Adding nodes with the bipartite attribute
    G.add_nodes_from(plants, bipartite=0)
    G.add_nodes_from(impollinators, bipartite=1)

    # Addition of edges 
    # Create a list of tuples (source_node, destination_node, attributes)
    edges = []
    for p in plants:
        for i in impollinators:
            weight = df.loc[p, i]
            if weight > 0: # type: ignore
                edges.append((p, i, {'weight': weight}))
                
    G.add_edges_from(edges)
    
    partition_plant = [i for i, n in enumerate(list(G.nodes())) if G.nodes[n]['bipartite'] == 0]
    partition_pollinators = [i for i, n in enumerate(list(G.nodes())) if G.nodes[n]['bipartite'] == 1]


    return G, partition_plant, partition_pollinators,plants,impollinators

For every csv in data_wol_pollinators we create the graph associated and embedding of the nodes of the graph.

In [3]:
folder = r"C:\Users\Utente\Desktop\progetto vscode\data\data_wol_pollinators"
path_files = glob.glob(os.path.join(folder, "*.csv"))

# Creations of lists and set with every data 
all_graphs = []  #list of all graphs of the dataset
all_embeddings ={}  #list of all the embeddings of the corresponding graph
all_plants=set()  #set of the plants present across all graphs
all_pollinators=set()  #set of the pollinators present across all graphs

for f in path_files:
    g,ppl,ppo, plants,pollinators=create_bipartite_graph(f)
    id_grafo = os.path.basename(f).replace('.csv', '')
    g.graph['name'] = id_grafo
    g_e=node_embedding(g,[ppl,ppo],int(len(g.nodes())/4))
    all_graphs.append(g)
    all_embeddings[g]=g_e
    all_plants.update(plants)
    all_pollinators.update(pollinators)


In [65]:
b=list(all_plants)
b.sort()
pl=[[i.split()[0]] for i in b]

In [57]:
l1 = ['aa', 'bb', 'cc', 'ab']
l2 = ['a', 'b', 'c', 'b']

# 1. Raggruppiamo tutti gli elementi
diz_temporaneo = {}
for i, chiave in enumerate(pl):
    if chiave[0] not in diz_temporaneo:
        diz_temporaneo[chiave[0]] = []  # Crea una nuova lista se la chiave non esiste
    diz_temporaneo[chiave[0]].append(b[i])

# A questo punto diz_temporaneo sarà: {'a': ['aa'], 'b': ['bb', 'ab'], 'c': ['cc']}

# 2. Filtriamo solo le chiavi con più di un elemento usando una Dictionary Comprehension
diz_finale = {k: v for k, v in diz_temporaneo.items() if len(v) > 1}

print(diz_temporaneo)
print(len(diz_temporaneo))
print(len(diz_finale))
# Output: {'b': ['bb', 'ab']}

{'Abarema': ['Abarema brachystachya', 'Abarema jupunba'], 'Abelia': ['Abelia grandiflora', 'Abelia serrata', 'Abelia sp1 M_PL_054'], 'Abelmoschus': ['Abelmoschus manihot'], 'Abundance"': ['Abundance"'], 'Abutilon': ['Abutilon bedfordianum', 'Abutilon darwinii', 'Abutilon depauperatum', 'Abutilon terminale', 'Abutilon theophrasti'], 'Acacia': ['Acacia insulae-iacobi', 'Acacia macracantha'], 'Acaena': ['Acaena pinn', 'Acaena pinnatifida'], 'Acanthus': ['Acanthus spinosus'], 'Acer': ['Acer carpinifolium', 'Acer insulae', 'Acer japonicum', 'Acer negundo', 'Acer rufinerve', 'Acer saccharinum', 'Acer shirasawanum', 'Acer ukurunduense'], 'Achillea': ['Achillea alpina ', 'Achillea millefolium'], 'Achyranthes': ['Achyranthes bidentata'], 'Achyrocline': ['Achyrocline satureoides'], 'Aciphylla': ['Aciphylla glacialis', 'Aciphylla scott-thompsoni', 'Aciphylla simplicifolia', 'Aciphylla subflavellata'], 'Acnistus': ['Acnistus arborescens'], 'Aconitum': ['Aconitum columbianum', 'Aconitum japonicum',

In [ ]:
island_mapping = {
    # Continental Island 
    'Amami-Ohsima Island, Japan': 'continental island',
    'Arima Valley': 'continental island',
    "Arthur's Pass, New Zealand": 'continental island',
    'Ashu, Kyoto, Japan': 'continental island',
    'Bristol, England': 'continental island',
    'Cass, New Zealand': 'continental island',
    'Chiloe, Chile': 'continental island',
    'Craigieburn, New Zealand': 'continental island',
    'Hazen Camp, Ellesmere Island, Canada': 'continental island',
    'Hickling, Norfolk, UK': 'continental island',
    'Kibune, Kyoto, Japan': 'continental island',
    'Kyoto City, Japan': 'continental island',
    'Matamata': 'continental island',
    'Melville Island, Canada': 'continental island',
    'Morne Seychellois National Park, Mahé': 'continental island',
    'Mt. Kushigata, Yamanashi Pref., Japan': 'continental island',
    'Mt. Yufu, Japan': 'continental island',
    'Nakaikemi marsh, Fukui Prefecture, Japan': 'continental island',
    'Shelfanger, Norfolk, UK': 'continental island',
    'Tundra, Greenladn': 'continental island',
    'Uummannaq Island, Greenland': 'continental island',
    'Zackenberg': 'continental island',
    
    # Oceanic Island 
    'Black River Gorges National Park, Mauritius': 'oceanic island',
    'Flores, Açores': 'oceanic island',
    'Galapagos': 'oceanic island',
    'Garajonay, Gomera, Spain': 'oceanic island',
    'Mauritius Island': 'oceanic island',
    'Morant Point, Jamaica': 'oceanic island',
    'Puerto Villamil, Isabela Island, Galapagos': 'oceanic island',
    'Syndicate, Dominica': 'oceanic island',
    'Tenerife, Canary Islands': 'oceanic island',
    'Windsor, The Cockpit Country, Jamaica': 'oceanic island'
}

In [ ]:
region_mapping = {
    # NEOTROPICAL (South America, Central America, and the Caribbean) 
    'Amarante, Pampas, Argentina': 'Neotropical',
    'Antonio Porto Jatai': 'Neotropical',
    'Arima Valley': 'Neotropical',
    'Atlantic Forest, high elevation': 'Neotropical',
    'Atlantic Forest, low elevation': 'Neotropical',
    'Atlantic Forest, mid elevation': 'Neotropical',
    'Calarca, Quindio': 'Neotropical',
    'Cambucá, Ubatuba': 'Neotropical',
    'Canaima Nat. Park, Venezuela': 'Neotropical',
    'Chiloe, Chile': 'Neotropical',
    'Cinco Cerros, Pampas, Argentina': 'Neotropical',
    'Cordón del Cepo, Chile': 'Neotropical',
    'Diamantina': 'Neotropical',
    'Difuntito, Pampas, Argentina': 'Neotropical',
    'Difuntos, Pampas, Argentina': 'Neotropical',
    'El Morro, Pampas, Argentina': 'Neotropical',
    'El Triunfo 1, Biosphere Reserve': 'Neotropical',
    'El Triunfo 2, Biosphere Reserve': 'Neotropical',
    'Estacion de Biologia Chamela, Jalisco': 'Neotropical',
    'Galapagos': 'Neotropical',
    'Guarico State, Venezuela': 'Neotropical',
    'La Barrosa, Pampas, Argentina': 'Neotropical',
    'La Brava, Pampas, Argentina': 'Neotropical',
    'La Chata, Pampas, Argentina': 'Neotropical',
    'La Paja, Pampas, Argentina': 'Neotropical',
    'Laguna Diamante, Mendoza, Argentina': 'Neotropical',
    'Mindo Lindo': 'Neotropical',
    'Morant Point, Jamaica': 'Neotropical',
    'Nahuel Huapi National Park, Argentina': 'Neotropical',
    'Nanegal 1': 'Neotropical',
    'Nanegal 2': 'Neotropical',
    'Parque Estadual Carlos Botelho': 'Neotropical',
    'Parque Nacional Chiribiquete': 'Neotropical',
    'Parque Nacional do Catimbau': 'Neotropical',
    'Picinguaba, Ubatuba': 'Neotropical',
    'Piedra Alta, Pampas, Argentina': 'Neotropical',
    'Puerto Villamil, Isabela Island, Galapagos': 'Neotropical',
    'Reserva Florestal Mata do Paraíso, Brazil': 'Neotropical',
    'Rio Blanco, Mendoza, Argentina': 'Neotropical',
    'Salento, Quindio': 'Neotropical',
    'Santa Virginia Field Station, Serra do Mar State Park': 'Neotropical',
    'Santuario de Flora y Fauna Galeras': 'Neotropical',
    'Serra da Mantiqueira, PNI, SE Brazil': 'Neotropical',
    'Serra do Cipo National Park': 'Neotropical',
    'Serra do Mar State Park': 'Neotropical',
    'Serra do Mar and Serra da Mantiqueira': 'Neotropical',
    'Serra do Pará, Brazil': 'Neotropical',
    'Syndicate, Dominica': 'Neotropical',
    'Unchog, Carpish Mountains': 'Neotropical',
    'Vigilancia, Pampas, Argentina': 'Neotropical',
    'Volcan, Pampas, Argentina': 'Neotropical',
    'Windsor, The Cockpit Country, Jamaica': 'Neotropical',

    # PALEARCTIC (Europe, North Africa, North and Central Asia)
    'Ashu, Kyoto, Japan': 'Palearctic',
    'Bristol, England': 'Palearctic',
    'Daphní, Athens, Greece': 'Palearctic',
    'Denmark': 'Palearctic',
    'Doñana Nat. Park, Spain': 'Palearctic',
    'Flores, Açores': 'Palearctic',
    'Garajonay, Gomera, Spain': 'Palearctic',
    'Hestehaven, Denmark': 'Palearctic',
    'Hickling, Norfolk, UK': 'Palearctic',
    'Isenbjerg': 'Palearctic',
    'Kibune, Kyoto, Japan': 'Palearctic',
    'Kyoto City, Japan': 'Palearctic',
    'Latnjajaure, Abisko, Sweden': 'Palearctic',
    'Mt. Kushigata, Yamanashi Pref., Japan': 'Palearctic',
    'Mt. Yufu, Japan': 'Palearctic',
    'Nakaikemi marsh, Fukui Prefecture, Japan': 'Palearctic',
    'Parc Natural del Cap de Creus': 'Palearctic',
    'Shelfanger, Norfolk, UK': 'Palearctic',
    'Tenerife, Canary Islands': 'Palearctic',

    # NEARTICA (North America and Greenland)
    'Brownfield, Illinois, USA': 'Nearctic',
    'Carlinville, Illinois, USA': 'Nearctic',
    'Central New Brunswick, Canada': 'Nearctic',
    'Hazen Camp, Ellesmere Island, Canada': 'Nearctic',
    'Highland temperate mosaic forest, Central Mexico': 'Nearctic',
    'Melville Island, Canada': 'Nearctic',
    'Montgomery County, Maryland, USA': 'Nearctic',
    'North Carolina, USA': 'Nearctic',
    'Ottawa, Canada': 'Nearctic',
    'Pikes Peak, Colorado, USA': 'Nearctic',
    'Tundra, Greenladn': 'Nearctic', # Ho mantenuto il typo 'Greenladn' per sicurezza del match
    'Uummannaq Island, Greenland': 'Nearctic',
    'Zackenberg': 'Nearctic',

    # AFROTROPICAL (Sub-Saharan Africa and the West Indies)
    'Black River Gorges National Park, Mauritius': 'Afrotropical',
    'KwaZulu-Natal region, South Africa': 'Afrotropical',
    'Mauritius Island': 'Afrotropical',
    'Morne Seychellois National Park, Mahé': 'Afrotropical',

    #  AUSTRALASIA (Australia, New Zealand)
    "Arthur's Pass, New Zealand": 'Australasian',
    'Cass, New Zealand': 'Australasian',
    'Craigieburn, New Zealand': 'Australasian',
    'Matamata': 'Australasian',
    'Snowy Mountains, Australia': 'Australasian',

    # INDONESIAN (Southeast Asia)
    'Amami-Ohsima Island, Japan': 'Indomalayan'
}

In [ ]:
metadata = pd.read_csv(os.path.join(folder, 'references.csv'))
metadata['island_type'] = metadata['Locality of Study'].map(island_mapping).fillna('mainland')
metadata['region'] = metadata['Locality of Study'].map(region_mapping)

metadata['ID'] = metadata['ID'].astype(str)

ordinamento_ids = [str(grafo.graph['name']) for grafo in all_graphs]

metadata.set_index('ID', inplace=True)

metadata = metadata.reindex(ordinamento_ids)
metadata = metadata[:-1]

print(metadata.head())

Create a matrix with elements all the distances calculated for all the pairsof graphs.

In [ ]:
n_networks = len(all_graphs)-1
dist_matrix = np.zeros((n_networks, n_networks))

for i in range(n_networks):
    for j in range(i + 1, n_networks):
        grafo_i = all_graphs[i]
        grafo_j = all_graphs[j]
        
        # lists with embedding matrices
        X_list = all_embeddings[grafo_i]
        Y_list = all_embeddings[grafo_j]
        
        # calculate the Wasserstein distance between the classes
        dist_val, _, _ = w2_distance_classes(X_list, Y_list)
        
        # simmetric matrix
        dist_matrix[i, j] = dist_val
        dist_matrix[j, i] = dist_val

# scikit-bio
ids_stringa = [str(x) for x in metadata.index.tolist()]
dm = DistanceMatrix(dist_matrix, ids=ids_stringa)


PERMANOVA (Permutational Multivariate Analysis of Variance) is a non-parametric statistical test used to compare the overall composition of two or more groups.

It evaluates whether the groups are significantly different from each other based on a calculated distance matrix. Instead of assuming that the data follows a normal distribution, it relies on random permutations to calculate the statistical significance (p-value).

PERMANOVA: island type (Oceanic, Continental, Mainland)

In [ ]:
island_types = metadata['island_type'].values
permanova_res = permanova(dm, island_types, permutations=999) #skbio.stats.distance.permanova(distmat, grouping, column=None, permutations=999, seed=None)

print("PERMANOVA: islands type")
print(permanova_res)

 The PERMANOVA test shows a highly significant difference
 between the 3 island types. We can conclude that the island type strongly influences 
 the overall structure of our networks.

The Mantel test is a statistical tool used to evaluate the correlation between two separate distance matrices that describe the same set of objects.
It checks if the distances between pairs of items in one matrix are related to the distances between the exact same items in a second matrix. Because the values within distance matrices are not independent, the test uses random permutations to calculate the statistical significance (p-value) of the correlation.

 Mantel Test: latitude


In [ ]:

lat_dist = squareform(pdist(metadata[['Latitude']].values, metric='euclidean'))
lat_dm = DistanceMatrix(lat_dist, ids=ids_stringa)
corr, p_val, _ = mantel(dm, lat_dm, method='spearman') #skbio.stats.distance.mantel(x, y, method='pearson', permutations=999, alternative='two-sided', strict=True, lookup=None, seed=None)
print(f"MANTEL TEST: Latitude \nR: {corr:.3f}, p-value: {p_val:.3f}")


The Mantel test shows no significant correlation
between network differences and latitude. This suggests that geographic 
distance does not influence the network structure.

The ANOSIM (Analysis of Similarities) test is a non-parametric statistical method used to evaluate whether there is a significant difference in composition between two or more groups.
Similar to PERMANOVA, it operates on a distance matrix. However, instead of using the raw distance values, ANOSIM compares the ranks of the distances. It evaluates if the ranked distances between different groups are significantly greater than the ranked distances within the same groups, producing an R statistic and a p-value.

ANOSIM: regions

In [ ]:
if 'region' in metadata.columns:
    regions = metadata['region'].values
    anosim_res = anosim(dm, regions, permutations=999) #skbio.stats.distance.anosim(distmat, grouping, column=None, permutations=999, seed=None)

    print("ANOSIM: biogeographic region")
    print(anosim_res)

The ANOSIM test shows no significant difference
across the 6 biogeographic regions. This indicates that the biogeographic 
region does not drive the structural composition of the networks.

The analysis reveal that network composition is significantly 
 driven by 'island type'. 
 
 In contrast, spatial and geographical factors, such as latitudinal distance
 and broad biogeographic regions, have no significant effect 
 on the structural similarity of the networks.